# 📘 Modèle Pyomo généré automatiquement

## 📦 Imports

In [43]:
from pyomo.environ import *
from pyomo.opt import SolverFactory
import pandas as pd

## 🔹 Model

In [44]:
from pyomo.environ import *

model = ConcreteModel()

## 🔹 Sets

In [45]:
model.MACHINES = Set(initialize=[1, 2])
model.PRODUITS = Set(initialize=['remorcage', 'stabilisateur'])
model.ARC = Set(dimen=2, initialize=[(i,j) for i in model.MACHINES for j in model.PRODUITS])

## 🔹 Parameters

In [46]:
model.disponibilite = Param(model.MACHINES, initialize={1: 16.0, 2: 15.0}, within=NonNegativeReals)
model.gain = Param(model.PRODUITS, initialize={'remorcage': 130.0, 'stabilisateur': 150.0}, within=NonNegativeReals)
model.temps = Param(model.MACHINES, model.PRODUITS, initialize={(1, 'remorcage'): 3.2, (1, 'stabilisateur'): 2.4, (2, 'remorcage'): 2.0, (2, 'stabilisateur'): 3.0}, within=NonNegativeReals)

## 🔹 Variables

In [47]:
model.x = Var(model.PRODUITS, domain=NonNegativeReals)

## 🔹 Constraints

In [48]:
model.c_for_0 = ConstraintList()
for m in model.MACHINES:
    model.c_for_0.add(sum(model.temps[m,p] * model.x[p] for p in model.PRODUITS) <= model.disponibilite[m])

## 🔹 Objective

In [49]:
model.obj = Objective(expr=sum(model.gain[p] * model.x[p] for p in model.PRODUITS), sense=maximize)

## ⚙️ Résolution du modèle

In [50]:
solver = SolverFactory('highs')
result = solver.solve(model, tee=True)

print('✅ Solver status:', result.solver.status)
print('✅ Termination condition:', result.solver.termination_condition)

✅ Solver status: ok
✅ Termination condition: optimal


## 🎯 Valeur de la fonction objective

In [51]:
for obj in model.component_objects(Objective, active=True):
    print(f'Objectif: {obj.name}')
    print(f'Valeur optimale: {obj():.4f}')
    print(f'Sens: {"Minimisation" if obj.sense == minimize else "Maximisation"}')

Objectif: obj
Valeur optimale: 825.0000
Sens: Maximisation


## 📊 Valeurs optimales des variables

In [52]:
# Extraction des résultats dans un DataFrame
results_data = []
for v in model.component_objects(Var, active=True):
    for index in v:
        results_data.append({
            'Variable': v.name,
            'Index': str(index) if index != None else '-',
            'Valeur': v[index].value
        })

df_results = pd.DataFrame(results_data)
# Filtrer les valeurs non-nulles pour plus de clarté
df_results = df_results[df_results['Valeur'].notna()]
df_results = df_results[df_results['Valeur'] != 0]
df_results.style.format({'Valeur': '{:.4f}'}).set_caption('Variables de décision optimales')

,Variable,Index,Valeur
0,x,remorcage,2.5000
1,x,stabilisateur,3.3333
